## Imports



In [1]:

import pandas as pd
import geopandas as gpd
import numpy as np
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pytensor.tensor as pt
import libpysal
from statsmodels.stats.outliers_influence import variance_inflation_factor

### Two approaches to load files into memory

#### Google Colab

In [ ]:
# The following code is useful to install quickly cmdstanpy and mount drive
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/updated_data.geojson"
df = gpd.read_file(DATA_PATH)

#### Local runtime

In [2]:
DATA_PATH = "../data/updated_data_v2.geojson"
df = gpd.read_file(DATA_PATH)

## Preprocessing

In [3]:
ecec_covariates = ["ecec_participation", "coverage", "public_coverage", "private_coverage", "per_capita_public_expenditure",
                   "per_capita_user_contribution", "other_percapita_public_expenditure", "other_percapita_user_contrib",
                   "other_per_capita_expenditure", "ecec_diffusion"]

target = "fem_empl_rate"

In [4]:
to_drop = ecec_covariates
df_noecec = df.select_dtypes(include='number').drop(columns=to_drop)

n_obs, n_feat = df_noecec.shape
print(f"\nNumber of observations: {n_obs}, number of predictors: {n_feat}")


Number of observations: 107, number of predictors: 41


# Variable Selection

### Celeste's thesis: <br>
**Due to high correlation with women employment**:
- empl_rate
- unempl_rate
- inact_rate
- young_neet
- part_rate

**Due to high correlation with other variables**:
- male_education rate &rarr; kept fem_education_rate
- net_migration_rat, adj_net_migration_rate &rarr; graduate_mobility_rate
- GDP &rarr; advanced_business_empl, graduate_mobility_rate
- fertility_rate, dependency_index &rarr; ageing index

**Following's Lara's suggestions**:
- fem_house_rate &rarr; high correlation with fem_empl_rate
- house_price &rarr; high correlation with GDP
- urban_type, area_km2, internet_coverage_rate, green_m2_capita, eco_urban_index


## Correlation Based Selection

### Corelation with women employment rate

In [5]:
irrelevant_vars = ["urban_type", "area_km2", "internet_coverage_rate", "green_m2_capita", "eco_urban_index", "fem_house_rate","house_price"]
df.drop(columns=irrelevant_vars, inplace=True)

In [6]:
y_corr_p = df_noecec.corr(method='pearson')[target].sort_values(ascending=False)
y_high_p =y_corr_p[(y_corr_p > 0.85) | (y_corr_p < -0.85)]

y_corr_s = df_noecec.corr(method='spearman')[target].sort_values(ascending=False)
y_high_s =y_corr_s[(y_corr_s > 0.85) | (y_corr_s < -0.85)]

In [7]:
high_s = [k for k in y_high_s.index if k != target]
high_p = [k for k in y_high_p.index if k != target]
high_corr = high_s + [k for k in high_p if k not in high_s]
print(f"Following variables are dropped:\n {high_corr}")

Following variables are dropped:
 ['empl_rate', 'empl_gap', 'part_rate', 'inact_rate', 'inactive_women_20_64', 'mal_empl_rate', 'net_migration_rate', 'unempl_rate', 'young_neet']


In [8]:
df_reduced = df.drop(columns=high_corr)

In [9]:
dfr_noecec = df_reduced.drop(columns=ecec_covariates)

In [11]:
def high_correlated_pairs(df, threshold):

    corr_matrix = df.select_dtypes(include='number').corr(method='pearson') # changes with spearman
    # avoid duplicate pairs and self-correlation
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    high_corr_pairs = (
        upper.stack()
            .reset_index()
            .rename(columns={'level_0': 'var1', 'level_1': 'var2', 0: 'corr'})
    )

    high_corr_pairs = (
        high_corr_pairs[high_corr_pairs['corr'].abs() >= threshold]
        .sort_values(by='corr', key=np.abs, ascending=False)
    )
    
    return high_corr_pairs

In [12]:
# celeste used pearson corr and her results are slightly different
print(high_correlated_pairs(dfr_noecec, 0.8))

                       var1                    var2      corr
47             fem_edu_rate           male_edu_rate  0.951768
212          fertility_rate            ageing_index -0.872535
245         dependency_rate            ageing_index  0.851699
210          fertility_rate  avg_children_per_woman  0.847711
122                     GDP  graduate_mobility_rate  0.832262
171            foreign_rate  adj_net_migration_rate  0.823343
284  graduate_mobility_rate           fem_empl_rate  0.821337
265  adj_net_migration_rate  graduate_mobility_rate  0.816965
123                     GDP  advanced_business_empl  0.816609
141            n_large_comp  advanced_business_empl  0.813944
271  adj_net_migration_rate           fem_empl_rate  0.813880


From table above:
- rem male_edu_rate &rarr; keep fem_edu_rate
- rem fertility_rate and dependency_rate &rarr; keep ageing_index
- rem GDP, foreign_rate, adj_net_migration_rate, GDP, n_large_comp &rarr; keep graduate_mobility_rate

In [13]:
to_drop = ['male_edu_rate', 'fertility_rate','dependency_rate','GDP','foreign_rate','adj_net_migration_rate','n_large_comp']

In [14]:
df_reduced.drop(columns=to_drop, inplace=True)
print(df_reduced.shape)
print(df_reduced.columns)

(107, 33)
Index(['geo_point_2d', 'rip_name', 'reg_name', 'prov_name', 'population',
       'empl_growth', 'fem_edu_rate', 'service_empl_rate', 'n_self_empl',
       'n_members_family', 'elderly_in_care', 'first_birth_maternal_age',
       'avg_children_per_woman', 'ageing_index', 'public_transport_supply',
       'graduate_mobility_rate', 'per_capita_public_expenditure',
       'ecec_participation', 'coverage', 'per_capita_user_contribution',
       'advanced_business_empl', 'lifelong_learn', 'startup_ratio',
       'GERD_total_intramural', 'public_coverage', 'private_coverage',
       'other_percapita_public_expenditure', 'other_percapita_user_contrib',
       'other_per_capita_expenditure', 'fem_maj_empl_rate', 'ecec_diffusion',
       'fem_empl_rate', 'geometry'],
      dtype='object')


## ECEC covariate selection

In [15]:
print(f"The ECEC covariates are the following:\n")
for ecec_var in ecec_covariates:
    print(ecec_var)
high_correlated_pairs(df[ecec_covariates],0.80)


The ECEC covariates are the following:

ecec_participation
coverage
public_coverage
private_coverage
per_capita_public_expenditure
per_capita_user_contribution
other_percapita_public_expenditure
other_percapita_user_contrib
other_per_capita_expenditure
ecec_diffusion


,var1,var2,corr
40,other_percapita_public_expenditure,other_per_capita_expenditure,0.994723
3,ecec_participation,per_capita_public_expenditure,0.824716
10,coverage,private_coverage,0.806509


Issue: `ecec_participation` &rarr; influenced by expenditures: as if it were the result of per capita user contribution and per capita public expenditure (the two are highly correlated). <br>
Which ones should we keep? Maybe a conceptual model would be useful here. For the time being, let's reduce the number of covariated &rarr; keep only ecec_participation <br>
Likely that keeping track of expenditures makes more sense policy wise <br>
*reflections of the conceptual model &rarr; provision and organisation model shape usage, mediated by AAAQ usage impacts employment*

Maybe a different model to estimate the level of ECEC participation could be useful, rather than just include ecec_participation among other ECEC variables

In [16]:
ecec_drop = ['other_per_capita_expenditure','per_capita_public_expenditure','per_capita_user_contribution', 'coverage']

In [17]:
df_final = df_reduced.drop(columns=ecec_drop)

In [18]:
df_final.columns

Index(['geo_point_2d', 'rip_name', 'reg_name', 'prov_name', 'population',
       'empl_growth', 'fem_edu_rate', 'service_empl_rate', 'n_self_empl',
       'n_members_family', 'elderly_in_care', 'first_birth_maternal_age',
       'avg_children_per_woman', 'ageing_index', 'public_transport_supply',
       'graduate_mobility_rate', 'ecec_participation',
       'advanced_business_empl', 'lifelong_learn', 'startup_ratio',
       'GERD_total_intramural', 'public_coverage', 'private_coverage',
       'other_percapita_public_expenditure', 'other_percapita_user_contrib',
       'fem_maj_empl_rate', 'ecec_diffusion', 'fem_empl_rate', 'geometry'],
      dtype='object')

In [19]:
df_final['macro_area'] = df_final['rip_name'].apply(lambda x: 'Nord' if x in ['Nord-Ovest', 'Nord-Est'] else ('Centro' if x == 'Centro' else 'Sud'))
df_final['macro_area'].value_counts()

macro_area
Nord      47
Sud       38
Centro    22
Name: count, dtype: int64

In [20]:
selected_ecec = set(ecec_covariates) - set(ecec_drop)

In [ ]:
def corr_text_factory(df, hue, method="pearson", fmt="{:+.2f}"):
    """
    Returns a function suitable for PairGrid.map_upper that:
      - prints global correlation once
      - prints per-group correlations (one line per group) using the hue color
    """
    global_corr = df.select_dtypes(include='number').corr(method=method)

    def corr_text(x, y, color=None, label=None, **kws):
        ax = plt.gca()

        xname, yname = x.name, y.name

        # Add global correlation once per axis/cell
        if not hasattr(ax, "_global_done"):
            r_all = global_corr.loc[xname, yname]
            ax.text(
                0.03, 0.90, f"all: {fmt.format(r_all)}",
                transform=ax.transAxes,
                ha="left", va="top",
                fontsize=10, fontweight="bold", color="black"
            )
            ax._global_done = True

        # Compute group correlation for the subset passed in (this call is per hue group)
        r_g = x.corr(y, method=method)
        if np.isnan(r_g):
            txt = f"{label}: NaN"
        else:
            txt = f"{label}: {fmt.format(r_g)}"

        # Place each group line at a different vertical position
        # We rely on the order of calls per axis to stack lines.
        if not hasattr(ax, "_line_i"):
            ax._line_i = 0
        y0 = 0.72 - 0.14 * ax._line_i
        ax.text(
            0.03, y0, txt,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=9, color=color
        )
        ax._line_i += 1

        # Clean up the cell look
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)

    return corr_text


In [ ]:
# I tried mimicing ggpairs, but I eventually done it in R
data = df_final.drop(columns=selected_ecec).copy()
hue_col = "macro_area"

num_vars = data.select_dtypes(include='number').columns.tolist()

sns.set_style("white")
g = sns.PairGrid(data=data, vars=num_vars, hue=hue_col, corner=False, height=2.2)
# content
g.map_lower(sns.scatterplot, s=12, alpha=0.6, edgecolor="none")
g.map_diag(sns.kdeplot, fill=False)
g.map_upper(corr_text_factory(data[num_vars + [hue_col]], hue=hue_col, method="pearson"))

# legend
g.add_legend(title=hue_col)
g.fig.tight_layout()
plt.show()

In [ ]:
g_no_ecec = sns.pairplot(df_final.drop(columns=selected_ecec), hue='macro_area')

In [ ]:
g_ecec = sns.pairplot(df_final[list(selected_ecec) + [target] + ['macro_area']], hue='macro_area')

## Check VIF


In [27]:
covariates.columns

Index(['population', 'empl_growth', 'fem_edu_rate', 'service_empl_rate',
       'n_self_empl', 'n_members_family', 'elderly_in_care',
       'first_birth_maternal_age', 'avg_children_per_woman', 'ageing_index',
       'public_transport_supply', 'graduate_mobility_rate',
       'ecec_participation', 'advanced_business_empl', 'lifelong_learn',
       'startup_ratio', 'GERD_total_intramural', 'public_coverage',
       'private_coverage', 'other_percapita_public_expenditure',
       'other_percapita_user_contrib', 'fem_maj_empl_rate', 'ecec_diffusion',
       'fem_empl_rate'],
      dtype='object')

In [28]:
import statsmodels.api as sm
covariates = df_final.select_dtypes(include="number").copy()
covariates = covariates.drop(columns=['fem_empl_rate'])
covariates = sm.add_constant(covariates, has_constant='add')


In [29]:
vif = pd.DataFrame({
    "variable": covariates.columns,
    "VIF": [variance_inflation_factor(covariates.values, i) for i in range(covariates.shape[1])]
})


In [31]:
vif.sort_values('VIF', ascending=False)

,variable,VIF
0,const,13835.031048
13,ecec_participation,9.140282
12,graduate_mobility_rate,8.102231
14,advanced_business_empl,7.346519
6,n_members_family,6.564543
10,ageing_index,6.548976
18,public_coverage,4.139228
8,first_birth_maternal_age,4.010305
3,fem_edu_rate,3.987515
20,other_percapita_public_expenditure,3.876044


## Bayesian Variable Selection

In [ ]:
df = df_final.reset_index(drop=True).copy()

X = df.select_dtypes(include='number').to_numpy()
y = df[target].to_numpy()

X_scaler = StandardScaler()
X_star = X_scaler.fit_transform(X)

y = y / 100
y_star = np.log(y / (1 - y))


n_obs, n_feat = X_star.shape
print(f"\nNumber of observations: {n_obs}, number of predictors: {n_feat}")

# spatial structure
w = libpysal.weights.Rook.from_dataframe(df)
W_sparse = w.sparse




In [ ]:
from scipy import sparse
assert sparse.issparse(W_sparse) 
assert W_sparse.shape == (n_obs, n_obs)

print("Diagonal max abs:", np.abs(W_sparse.diagonal()).max())

## Bayesian Lasso with spatial random effect

In [ ]:
with pm.Model() as bl_car_model:
    # --- data
    X_data = pm.Data("X_data", X_star)
    y_data = pm.Data("y_data", y_star)

    # --- LASSO components
    lambda_sq = pm.Gamma("lambda_sq", alpha=2.0, beta=1.0)
    lambda_ = pm.Deterministic("lambda", pm.math.sqrt(lambda_sq))
    sigma = pm.HalfNormal("sigma", sigma=1.0)
    beta = pm.Laplace("beta", mu=0.0, b=sigma / lambda_, shape=n_feat)
    intercept = pm.Normal("alpha", mu=0.0, sigma=10.0)

    # --- spatial random effects
    tau_c = pm.Gamma("tau_c", alpha=1.0, beta=1.0)
    alpha_c = pm.Beta("alpha_c", alpha=2.0, beta=2.0) # this is our rho

    phi = pm.CAR(
        "phi",
        mu=pt.zeros(n_obs),
        W=W_sparse,          # <<< scipy sparse CSR/CSC recommended
        alpha=alpha_c,
        tau=tau_c,
        shape=n_obs,
    )
    phi_centered = pm.Deterministic("phi_centered", phi - pt.mean(phi)) # they said to remove this

    # --- Likelihood ---
    mu = pm.math.dot(X_data, beta) + intercept + phi_centered # how can the model discern between spatial effect and intercept?
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_data)

    trace = pm.sample(draws=2000,
                      tune=2000,
                      target_accept=0.95,
                      return_inferencedata=True,
                      chains=4,
                      cores=4)

In [ ]:
summary_beta_95 = az.summary(trace, var_names=["beta"], hdi_prob=0.95)

summary_beta_95.index = selected_covariates

summary_beta_95["contains_zero"] = (
    (summary_beta_95["hdi_2.5%"] <= 0) &
    (summary_beta_95["hdi_97.5%"] >= 0)
)

summary_beta_95 = summary_beta_95.sort_values("mean", key=np.abs, ascending=False)

print(summary_beta_95[["mean", "hdi_2.5%", "hdi_97.5%", "contains_zero"]])

selected_covariates_df = summary_beta_95[~summary_beta_95["contains_zero"]].index.tolist()

In [ ]:
print(selected_covariates_df)

In [ ]:
beta_raw = trace.posterior["beta"].stack(samples=("chain", "draw")).values
if beta_raw.shape[0] == len(selected_covariates):
    beta_samples = beta_raw
else:
    beta_samples = beta_raw.T

n_feat = len(selected_covariates)

plt.figure(figsize=(12, 4))

for j in range(n_feat):
    sns.kdeplot(beta_samples[j, :], color="gray", alpha=0.3, linewidth=1)

for name in summary_beta_95.index[~summary_beta_95["contains_zero"]]:
    j = selected_covariates.index(name)
    sns.kdeplot(beta_samples[j, :], linewidth=2, label=name)

plt.legend()
plt.title("Posterior densities of β_j")
plt.xlabel("Values of β_j")
plt.yticks([])
plt.gca().tick_params(axis='y', length=0)
plt.show()

## Horseshoe prior

In [ ]:
with pm.Model() as hs_car_model:
    X_data = pm.Data("X_data", X_star)
    y_data = pm.Data("y_data", y_star)

    # Likelihood noise on logit scale
    sigma = pm.HalfNormal("sigma", sigma=1.0)

    # Horseshoe prior for beta (non-centered)
    lambda_local = pm.HalfCauchy("lambda_local", beta=1.0, shape=n_feat)
    tau_global = pm.HalfCauchy("tau_global", beta=1.0)

    beta_raw = pm.Normal("beta_raw", 0.0, 1.0, shape=n_feat)
    beta = pm.Deterministic("beta", beta_raw * tau_global * lambda_local)

    intercept = pm.Normal("alpha", 0.0, 5.0)

    # Proper CAR-like spatial effect with precision Q
    tau_c = pm.Gamma("tau_c", 1.0, 1.0)
    alpha_c = pm.Beta("alpha_c", 2.0, 2.0)

    phi = pm.CAR(
        "phi",
        mu=pt.zeros(n_obs),
        W=W_sparse,          # keep sparse!
        alpha=alpha_c,
        tau=tau_c,
        shape=n_obs
    )
    phi_centered = pm.Deterministic("phi_centered", phi - pt.mean(phi))

    mu = pm.math.dot(X_data, beta) + intercept + phi_centered

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_data)

    idata = pm.sample(
        draws=2000,
        tune=2000,
        target_accept=0.95,
        chains=4,
        cores=4,
        init="jitter+adapt_diag",
        return_inferencedata=True,
    )

In [ ]:
# HS results
az.summary(idata, var_names=["alpha", "sigma", "tau_global", "tau_c", "alpha_c", "beta"])
az.plot_trace(idata, var_names=["tau_global", "tau_c", "alpha_c"])

## Results

In [ ]:
#TODO check vif of variable selection

In [ ]:
final_covariates = selected_covariates_df + ecec_selected
print(f"{len(final_covariates)} selected covariates:")
print(final_covariates)

In [ ]:
freq_lasso_selection = ["coverage", "ecec_diffusion", "per_capita_public_expenditure", "ecec_participation", "per_capita_user_contribution", "ageing_index", "service_empl_rate", "n_members_family", "avg_children_per_woman", "fem_house_rate"]

In [ ]:
all_covariates = (set(final_covariates) | set(freq_lasso_selection))

# Build and print the table
print(f"{'covariate':40s} | {'in_final_covariates':20s} | {'in_freq_lasso':15s}")
print("-" * 85)

for cov in all_covariates:
    in_final = cov in final_covariates
    in_freq  = cov in freq_lasso_selection
    print(f"{cov:40s} | {str(in_final):20s} | {str(in_freq):15s}")